# LAB 3 — End-to-End Predictive Pipeline

ขยายจาก LAB 2 สู่ Pipeline:
**Data → Item Vector → Transformation → User Profile → User–Item Pair Features → Predictive Model → Ranking → Evaluation**

เพิ่มทั้ง:
- Relevance Classification
- Rating Regression
- Top-K Ranking Evaluation
- Beyond-Accuracy Evaluation

In [2]:
# ============================================================
# Part 0 — Environment & Dataset
# ============================================================
!pip -q install gradio gensim sentence-transformers transformers

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import math, random, warnings
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_auc_score, mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

BASE = Path("/content/drive/MyDrive/Colab Notebooks")
DATA_ROOT = BASE / "Dataset_Chapter3_4"

if (DATA_ROOT / "data").exists():
    DATA_DIR = DATA_ROOT / "data"
else:
    DATA_DIR = DATA_ROOT

print("BASE      =", BASE)
print("DATA_ROOT =", DATA_ROOT)
print("DATA_DIR  =", DATA_DIR)

items = pd.read_csv(DATA_DIR / "metadata.csv")
ratings = pd.read_csv(DATA_DIR / "ratings.csv")

rename_items = {
    "movie_id":"item_id", "id":"item_id",
    "movie_title":"title", "name":"title",
    "genres":"genre", "description":"synopsis",
    "plot":"synopsis", "summary":"synopsis"
}
for old,new in rename_items.items():
    if old in items.columns and new not in items.columns:
        items = items.rename(columns={old:new})

rename_r = {"uid":"user_id","user":"user_id","iid":"item_id","movie_id":"item_id","score":"rating"}
for old,new in rename_r.items():
    if old in ratings.columns and new not in ratings.columns:
        ratings = ratings.rename(columns={old:new})

required_i = ["item_id","title","genre","synopsis"]
required_r = ["user_id","item_id","rating"]
assert all(c in items.columns for c in required_i), items.columns.tolist()
assert all(c in ratings.columns for c in required_r), ratings.columns.tolist()

items["item_id"] = items["item_id"].astype(str)
ratings["item_id"] = ratings["item_id"].astype(str)
ratings["user_id"] = ratings["user_id"].astype(str)
ratings["rating"] = pd.to_numeric(ratings["rating"])

items["genre"] = items["genre"].fillna("").astype(str)
items["synopsis"] = items["synopsis"].fillna("").astype(str)
items["text_content"] = (items["genre"] + " " + items["synopsis"]).str.strip()

ITEM_INDEX = {iid:i for i,iid in enumerate(items["item_id"])}
N_USERS = max(1, ratings["user_id"].nunique())
POPULARITY = ratings.groupby("item_id")["user_id"].nunique().to_dict()

print(f"Items={len(items)}, Users={ratings['user_id'].nunique()}, Ratings={len(ratings)}")
display(items[["item_id","title","genre","synopsis"]].head())

Mounted at /content/drive
BASE      = /content/drive/MyDrive/Colab Notebooks
DATA_ROOT = /content/drive/MyDrive/Colab Notebooks/Dataset_Chapter3_4
DATA_DIR  = /content/drive/MyDrive/Colab Notebooks/Dataset_Chapter3_4/data
Items=12, Users=8, Ratings=32


,item_id,title,genre,synopsis
0,i1,Toy Story 5,Animation|Family,A cowboy toy and a space ranger toy lead the t...
1,i2,Inside Out 2,Animation|Family,"Inside the mind of a teenage girl, five emotio..."
2,i3,The Wild Robot,Animation|Sci-Fi,A shipwrecked robot boots up on a wild island ...
3,i4,It Ends with Us,Drama|Romance,A florist falls in love with a charming surgeo...
4,i5,Wicked,Fantasy|Musical,A green-skinned woman with a gift for magic jo...


In [3]:
# ============================================================
# Core Functions
# Vector Transformation + 3 User Profiles + Ranking + Metrics
# ============================================================

LIKED_THRESHOLD = 4.0
DISLIKED_THRESHOLD = 2.0

def transform_vectors(X, mode="Raw+L2"):
    X = np.asarray(X, dtype=float)
    if mode == "Raw+L2":
        return normalize(X)
    if mode == "Center+L2":
        return normalize(X - X.mean(axis=0, keepdims=True))
    if mode == "Standard+L2":
        return normalize(StandardScaler().fit_transform(X))
    raise ValueError(mode)

def _unit(v):
    v=np.asarray(v,dtype=float)
    n=np.linalg.norm(v)
    return v if n==0 else v/n

def profile_positive_only(user_id, X, ratings_df):
    r=ratings_df[ratings_df["user_id"].astype(str)==str(user_id)]
    liked=r[r["rating"]>=LIKED_THRESHOLD]
    idx=[ITEM_INDEX[str(i)] for i in liked["item_id"] if str(i) in ITEM_INDEX]
    if not idx:
        idx=[ITEM_INDEX[str(i)] for i in r["item_id"] if str(i) in ITEM_INDEX]
    return _unit(X[idx].mean(axis=0)) if idx else np.zeros(X.shape[1])

def profile_rating_centered(user_id, X, ratings_df):
    r=ratings_df[ratings_df["user_id"].astype(str)==str(user_id)].copy()
    r=r[r["item_id"].astype(str).isin(ITEM_INDEX)]
    if r.empty: return np.zeros(X.shape[1])
    idx=np.array([ITEM_INDEX[str(i)] for i in r["item_id"]])
    w=r["rating"].astype(float).values - float(r["rating"].mean())
    denom=np.abs(w).sum()
    p=X[idx].mean(axis=0) if denom<=1e-12 else (X[idx]*w[:,None]).sum(axis=0)/denom
    return _unit(p)

def profile_rocchio(user_id, X, ratings_df, alpha=1.0, beta=1.0):
    r=ratings_df[ratings_df["user_id"].astype(str)==str(user_id)]
    liked=r[r["rating"]>=LIKED_THRESHOLD]
    disliked=r[r["rating"]<=DISLIKED_THRESHOLD]
    li=[ITEM_INDEX[str(i)] for i in liked["item_id"] if str(i) in ITEM_INDEX]
    di=[ITEM_INDEX[str(i)] for i in disliked["item_id"] if str(i) in ITEM_INDEX]
    pos=X[li].mean(axis=0) if li else np.zeros(X.shape[1])
    neg=X[di].mean(axis=0) if di else np.zeros(X.shape[1])
    p=alpha*pos-beta*neg
    if np.linalg.norm(p)==0:
        hi=[ITEM_INDEX[str(i)] for i in r["item_id"] if str(i) in ITEM_INDEX]
        p=X[hi].mean(axis=0) if hi else p
    return _unit(p)

PROFILE_METHODS = {
    "Positive-only": profile_positive_only,
    "Rating-centered": profile_rating_centered,
    "Rocchio": profile_rocchio,
}

def get_profile(user_id, X, ratings_df, method, alpha=1.0, beta=1.0):
    if method=="Rocchio":
        return profile_rocchio(user_id,X,ratings_df,alpha,beta)
    return PROFILE_METHODS[method](user_id,X,ratings_df)

def rank_user(user_id, X, ratings_df, method, k=None, candidates=None, alpha=1.0, beta=1.0):
    p=get_profile(user_id,X,ratings_df,method,alpha,beta)
    score=cosine_similarity(p.reshape(1,-1),X).ravel()
    seen=set(ratings_df.loc[ratings_df["user_id"].astype(str)==str(user_id),"item_id"].astype(str))
    if candidates is None:
        candidates=[iid for iid in items["item_id"] if iid not in seen]
    ranked=sorted(candidates,key=lambda iid:score[ITEM_INDEX[iid]],reverse=True)
    return ranked[:k] if k else ranked, score, p

# Accuracy / Ranking Metrics
def precision_at_k(recs, rel, k):
    return len(set(recs[:k]) & set(rel))/k if k else 0.0
def recall_at_k(recs, rel, k):
    rel=set(rel); return len(set(recs[:k])&rel)/len(rel) if rel else 0.0
def f1_at_k(recs, rel, k):
    p=precision_at_k(recs,rel,k); r=recall_at_k(recs,rel,k)
    return 0.0 if p+r==0 else 2*p*r/(p+r)
def hr_at_k(recs, rel, k):
    return float(bool(set(recs[:k])&set(rel)))
def reciprocal_rank(recs, rel):
    rel=set(rel)
    for rank,iid in enumerate(recs,1):
        if iid in rel: return 1.0/rank
    return 0.0
def ap_at_k(recs, rel, k):
    rel=set(rel); hits=0; s=0.0
    for i,iid in enumerate(recs[:k],1):
        if iid in rel:
            hits+=1; s+=hits/i
    return s/min(len(rel),k) if rel else 0.0
def ndcg_at_k(recs, rel, k):
    rel=set(rel)
    dcg=sum(1/math.log2(i+2) for i,iid in enumerate(recs[:k]) if iid in rel)
    idcg=sum(1/math.log2(i+2) for i in range(min(len(rel),k)))
    return dcg/idcg if idcg else 0.0

# Beyond-Accuracy Metrics
def coverage(all_recs):
    return len(set(all_recs))/len(items) if len(items) else 0.0

def gini_coefficient(all_recs):
    counts=pd.Series(all_recs).value_counts()
    x=np.array([counts.get(iid,0) for iid in items["item_id"]],dtype=float)
    if x.sum()==0: return 0.0
    x=np.sort(x); n=len(x)
    return float((2*np.sum(np.arange(1,n+1)*x)/(n*x.sum()))-(n+1)/n)

def ild(item_ids, X):
    idx=[ITEM_INDEX[i] for i in item_ids if i in ITEM_INDEX]
    if len(idx)<2: return 0.0
    D=1-cosine_similarity(X[idx])
    v=D[np.triu_indices_from(D,k=1)]
    return float(v.mean()) if len(v) else 0.0

def novelty(item_ids):
    vals=[]
    for iid in item_ids:
        pop=max(1,POPULARITY.get(iid,1))
        vals.append(-math.log2(pop/N_USERS))
    return float(np.mean(vals)) if vals else 0.0

def calibration_genre(user_id, rec_ids, ratings_df):
    hist=ratings_df[ratings_df["user_id"].astype(str)==str(user_id)]
    hist_genres=[]
    for iid in hist["item_id"].astype(str):
        if iid in ITEM_INDEX:
            hist_genres += [g.strip() for g in str(items.iloc[ITEM_INDEX[iid]]["genre"]).replace("|",",").split(",") if g.strip()]
    rec_genres=[]
    for iid in rec_ids:
        rec_genres += [g.strip() for g in str(items.iloc[ITEM_INDEX[iid]]["genre"]).replace("|",",").split(",") if g.strip()]
    if not hist_genres or not rec_genres: return np.nan
    cats=sorted(set(hist_genres)|set(rec_genres))
    hp=np.array([hist_genres.count(c) for c in cats],float); hp/=hp.sum()
    rp=np.array([rec_genres.count(c) for c in cats],float); rp/=rp.sum()
    return float(1-0.5*np.abs(hp-rp).sum())

def loo_folds(ratings_df, positive_threshold=4.0):
    folds=[]
    for uid,g in ratings_df.groupby("user_id"):
        pos=g[g["rating"]>=positive_threshold]
        if pos.empty: continue
        hold=pos.sort_values("rating").iloc[-1]
        folds.append((str(uid),ratings_df.drop(index=hold.name),str(hold["item_id"])))
    return folds

def evaluate_config(X, profile_method, k=3, alpha=1.0, beta=1.0, folds=None,
                    candidate_mode="Full Catalog", n_neg=20, seed=42):
    folds=folds or loo_folds(ratings)
    rng=np.random.default_rng(seed)
    rows=[]; global_recs=[]
    for uid,train,heldout in folds:
        seen=set(train.loc[train["user_id"].astype(str)==uid,"item_id"].astype(str))
        all_c=[iid for iid in items["item_id"] if iid not in seen]
        if heldout not in all_c: all_c.append(heldout)
        if candidate_mode=="Sampled Negatives":
            neg=[x for x in all_c if x!=heldout]
            sampled=list(rng.choice(neg,size=min(n_neg,len(neg)),replace=False)) if neg else []
            candidates=[heldout]+sampled
        else:
            candidates=all_c
        ranked,score,_=rank_user(uid,X,train,profile_method,candidates=candidates,alpha=alpha,beta=beta)
        topk=ranked[:k]; global_recs += topk
        y=np.array([1 if iid==heldout else 0 for iid in ranked])
        ys=np.array([score[ITEM_INDEX[iid]] for iid in ranked])
        auc=roc_auc_score(y,ys) if len(np.unique(y))==2 else np.nan
        rows.append({
            "user_id":uid,"heldout":heldout,
            f"Precision@{k}":precision_at_k(topk,[heldout],k),
            f"Recall@{k}":recall_at_k(topk,[heldout],k),
            f"F1@{k}":f1_at_k(topk,[heldout],k),
            f"HR@{k}":hr_at_k(topk,[heldout],k),
            "MRR":reciprocal_rank(ranked,[heldout]),
            f"MAP@{k}":ap_at_k(topk,[heldout],k),
            f"nDCG@{k}":ndcg_at_k(topk,[heldout],k),
            "AUC":auc,"ILD":ild(topk,X),"Novelty":novelty(topk),
            "Calibration":calibration_genre(uid,topk,train),
            "TopK":" | ".join(topk)
        })
    detail=pd.DataFrame(rows)
    summary=detail.select_dtypes(include=np.number).mean().to_dict() if len(detail) else {}
    summary["Coverage"]=coverage(global_recs)
    summary["Gini"]=gini_coefficient(global_recs)
    return pd.Series(summary), detail

## Part 1 — Representation และ Transformation

In [4]:
from sentence_transformers import SentenceTransformer
sbert=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
X_raw=sbert.encode(items["text_content"].tolist(),show_progress_bar=True)
X=transform_vectors(X_raw,"Raw+L2")
print("Item Vector Shape:",X.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Item Vector Shape: (12, 384)


## Part 2 — Train/Test Split ต่อ User
ใช้ random holdout เป็นตัวอย่าง หากมี timestamp สามารถเปลี่ยนเป็น temporal split ได้

In [5]:
rng=np.random.default_rng(42)
train_idx=[]; test_idx=[]
for uid,g in ratings.groupby("user_id"):
    ids=g.index.to_numpy()
    if len(ids)<2:
        train_idx += list(ids); continue
    ntest=max(1,int(round(0.25*len(ids))))
    chosen=set(rng.choice(ids,size=ntest,replace=False))
    test_idx += list(chosen)
    train_idx += [i for i in ids if i not in chosen]
train_r=ratings.loc[train_idx].copy()
test_r=ratings.loc[test_idx].copy()
print("Train:",len(train_r),"Test:",len(test_r))

Train: 24 Test: 8


## Part 3 — User–Item Pair Features
สร้าง User Profile จาก **train history เท่านั้น** เพื่อลด leakage  
Features = cosine(profile,item) + |profile-item| + profile*item

In [6]:
PROFILE_FOR_MODEL="Rating-centered"

def pair_features(user_id,item_id,train_df,X):
    p=get_profile(user_id,X,train_df,PROFILE_FOR_MODEL)
    v=X[ITEM_INDEX[str(item_id)]]
    cos=float(np.dot(p,v)/max(np.linalg.norm(p)*np.linalg.norm(v),1e-12))
    return np.concatenate([[cos],np.abs(p-v),p*v])

def build_pair_dataset(df,train_history,X):
    F=[]; y_cls=[]; y_reg=[]; meta=[]
    for _,r in df.iterrows():
        iid=str(r["item_id"]); uid=str(r["user_id"])
        if iid not in ITEM_INDEX: continue
        F.append(pair_features(uid,iid,train_history,X))
        y_reg.append(float(r["rating"]))
        y_cls.append(int(float(r["rating"])>=4))
        meta.append((uid,iid))
    return np.asarray(F),np.asarray(y_cls),np.asarray(y_reg),meta

F_train,yc_train,yr_train,meta_train=build_pair_dataset(train_r,train_r,X)
F_test,yc_test,yr_test,meta_test=build_pair_dataset(test_r,train_r,X)
print(F_train.shape,F_test.shape)

(24, 769) (8, 769)


## Part 4 — Predictive Models
Classifier probability ใช้เป็น ranking score; Rating regression ประเมินด้วย MAE/RMSE

In [7]:
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

classifiers={
    "LogisticRegression":LogisticRegression(max_iter=2000,class_weight="balanced"),
    "RandomForestClassifier":RandomForestClassifier(n_estimators=200,random_state=42,class_weight="balanced")
}
regressors={
    "Ridge":Ridge(alpha=1.0),
    "RandomForestRegressor":RandomForestRegressor(n_estimators=200,random_state=42)
}

cls_rows=[]
for name,m in classifiers.items():
    m.fit(F_train,yc_train)
    prob=m.predict_proba(F_test)[:,1]
    pred=(prob>=0.5).astype(int)
    cls_rows.append({
        "Model":name,
        "Accuracy":accuracy_score(yc_test,pred),
        "Precision":precision_score(yc_test,pred,zero_division=0),
        "Recall":recall_score(yc_test,pred,zero_division=0),
        "F1":f1_score(yc_test,pred,zero_division=0),
        "AUC":roc_auc_score(yc_test,prob) if len(np.unique(yc_test))==2 else np.nan
    })
display(pd.DataFrame(cls_rows).round(4))

reg_rows=[]
for name,m in regressors.items():
    m.fit(F_train,yr_train)
    pred=m.predict(F_test)
    reg_rows.append({
        "Model":name,
        "MAE":mean_absolute_error(yr_test,pred),
        "RMSE":mean_squared_error(yr_test,pred)**0.5
    })
display(pd.DataFrame(reg_rows).round(4))

,Model,Accuracy,Precision,Recall,F1,AUC
0,LogisticRegression,0.625,0.6250,1.0,0.7692,0.2
1,RandomForestClassifier,0.500,0.5714,0.8,0.6667,0.2


,Model,MAE,RMSE
0,Ridge,1.2616,1.4215
1,RandomForestRegressor,1.2256,1.3396


## Part 5 — Predictive Ranking + Chapter 2 Metrics
ใช้ probability ของ classifier จัดอันดับ candidate แล้วประเมิน Accuracy/Ranking และ Beyond-Accuracy

In [8]:
best_cls=classifiers["LogisticRegression"]

def predictive_rank(uid,model,X,train_df):
    seen=set(train_df.loc[train_df["user_id"].astype(str)==str(uid),"item_id"].astype(str))
    cand=[iid for iid in items["item_id"] if iid not in seen]
    if not cand: return [],{}
    F=np.vstack([pair_features(uid,iid,train_df,X) for iid in cand])
    prob=model.predict_proba(F)[:,1]
    score=dict(zip(cand,prob))
    return sorted(cand,key=lambda x:score[x],reverse=True),score

rows=[]; all_rec=[]
for uid,g in test_r.groupby("user_id"):
    rel=set(g.loc[g["rating"]>=4,"item_id"].astype(str))
    if not rel: continue
    ranked,score=predictive_rank(str(uid),best_cls,X,train_r)
    top3=ranked[:3]; all_rec += top3
    rows.append({
        "user_id":uid,
        "Precision@3":precision_at_k(top3,rel,3),
        "Recall@3":recall_at_k(top3,rel,3),
        "F1@3":f1_at_k(top3,rel,3),
        "HR@3":hr_at_k(top3,rel,3),
        "MRR":reciprocal_rank(ranked,rel),
        "MAP@3":ap_at_k(top3,rel,3),
        "nDCG@3":ndcg_at_k(top3,rel,3),
        "ILD":ild(top3,X),
        "Novelty":novelty(top3),
        "Calibration":calibration_genre(str(uid),top3,train_r)
    })
pred_rank_detail=pd.DataFrame(rows)
summary=pred_rank_detail.select_dtypes(include=np.number).mean().to_dict()
summary["Coverage"]=coverage(all_rec); summary["Gini"]=gini_coefficient(all_rec)
display(pred_rank_detail.round(4))
display(pd.Series(summary).to_frame("mean").round(4))

,user_id,Precision@3,Recall@3,F1@3,HR@3,MRR,MAP@3,nDCG@3,ILD,Novelty,Calibration
0,u1,0.3333,1.0,0.5,1.0,1.0000,1.0000,1.0,0.8677,1.2767,0.4167
1,u3,0.3333,1.0,0.5,1.0,1.0000,1.0000,1.0,0.6202,1.8050,0.5000
2,u4,0.0000,0.0,0.0,0.0,0.1667,0.0000,0.0,0.5705,1.6667,0.3333
3,u5,0.0000,0.0,0.0,0.0,0.2500,0.0000,0.0,0.6551,1.4150,0.1667
4,u7,0.3333,1.0,0.5,1.0,0.3333,0.3333,0.5,0.8293,1.1383,0.3333


,mean
Precision@3,0.2000
Recall@3,0.6000
F1@3,0.3000
HR@3,0.6000
MRR,0.5500
MAP@3,0.4667
nDCG@3,0.5000
ILD,0.7086
Novelty,1.4604
Calibration,0.3500


## Part 6 — Experiment: Transformation × Profile × Predictive Model
ควบคุม split เดิม เพื่อแยกผลของ feature representation ออกจาก algorithm

In [9]:
from sklearn.base import clone
pipeline_rows=[]
for trans in ["Raw+L2","Center+L2"]:
    Xt=transform_vectors(X_raw,trans)
    for prof in ["Positive-only","Rating-centered","Rocchio"]:
        PROFILE_FOR_MODEL=prof
        Ft,yct,_,_=build_pair_dataset(train_r,train_r,Xt)
        Fv,ycv,_,_=build_pair_dataset(test_r,train_r,Xt)
        for name,base_model in classifiers.items():
            m=clone(base_model); m.fit(Ft,yct)
            prob=m.predict_proba(Fv)[:,1]
            pipeline_rows.append({
                "Transformation":trans,"Profile":prof,"Model":name,
                "AUC":roc_auc_score(ycv,prob) if len(np.unique(ycv))==2 else np.nan
            })
pipeline_results=pd.DataFrame(pipeline_rows)
display(pipeline_results.round(4))

,Transformation,Profile,Model,AUC
0,Raw+L2,Positive-only,LogisticRegression,0.7333
1,Raw+L2,Positive-only,RandomForestClassifier,0.6333
2,Raw+L2,Rating-centered,LogisticRegression,0.2000
3,Raw+L2,Rating-centered,RandomForestClassifier,0.2000
4,Raw+L2,Rocchio,LogisticRegression,0.1333
5,Raw+L2,Rocchio,RandomForestClassifier,0.2667
6,Center+L2,Positive-only,LogisticRegression,0.7333
7,Center+L2,Positive-only,RandomForestClassifier,0.8000
8,Center+L2,Rating-centered,LogisticRegression,0.4000
9,Center+L2,Rating-centered,RandomForestClassifier,0.4000


## Part 7 — GUI: Predictive Recommendation

In [10]:
import gradio as gr
PROFILE_FOR_MODEL="Rating-centered"

def gui_predictive(user_id,k):
    ranked,score=predictive_rank(str(user_id),best_cls,X,train_r)
    top=ranked[:int(k)]
    hist=train_r[train_r["user_id"].astype(str)==str(user_id)].merge(
        items[["item_id","title","genre"]],on="item_id",how="left"
    )[["title","genre","rating"]].sort_values("rating",ascending=False)
    rec=[]
    for rank,iid in enumerate(top,1):
        r=items.iloc[ITEM_INDEX[iid]]
        rec.append({"Rank":rank,"Title":r["title"],"Genre":r["genre"],
                    "P(like)":round(float(score[iid]),4),"Synopsis":r["synopsis"]})
    return hist,pd.DataFrame(rec)

demo=gr.Interface(
    fn=gui_predictive,
    inputs=[gr.Dropdown(sorted(ratings["user_id"].unique()),label="User"),
            gr.Slider(1,5,3,1,label="Top-K")],
    outputs=[gr.Dataframe(label="Train History"),gr.Dataframe(label="Predictive Recommendations")],
    title="Chapter 3 — End-to-End Predictive Content Recommender"
)
demo.launch(debug=False)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e723f44576f4cc51dd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
